# Building an LLM Agent with `pydantic-ai`


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/the-matter-lab/cdmx-tutorials/blob/main/colab_agent.ipynb)


## What we will cover
- Run a plain LLM
- Add tools and let the agent decide when to call them
- See a simple multi-agent system in action
- Keep context across turns with `message_history`
- Add one practical guardrail with `UsageLimits`

## Before we start:

*   Create an OpenRouter account (https://openrouter.ai/)
*   Get a personal API key
*   List of free models available on OpenRouter (https://openrouter.ai/collections/free-models)



## 0) Setup

**Install dependencies**


In [ ]:
!pip install pydantic-ai logfire

import nest_asyncio
nest_asyncio.apply()
import ast
import json
import re
from dataclasses import dataclass
from datetime import datetime
from typing import Any
from zoneinfo import ZoneInfo

from pydantic_ai import Agent, RunContext
from pydantic_ai.usage import UsageLimitExceeded, UsageLimits
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

**Set your API key**

In [ ]:
import os

# os.environ["OPENROUTER_API_KEY"] ="<YOUR API KEY HERE>"
# os.environ["LOGFIRE_TOKEN"] = "<YOUR API KEY HERE>"


**Choose your LLM**

In [ ]:
LLM_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

**Observability with Logfire**

> We can use logfire to see real request/tool traces.






In [ ]:
import logfire

logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()
logfire.instrument_httpx(capture_all=True)

## 1) Baseline: no tools yet

This is just a chatbot — no actions, no tools. We will run this first to see how a plain model works.

In [ ]:
openrouter_provider = OpenAIProvider(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

agent = Agent(
    model=OpenAIChatModel(
        LLM_MODEL,
        provider=openrouter_provider),
    instructions=(
        "You are a tutorial workshop assistant for LLM agents."
    ),
)

query = ("Explain in 3 bullets: what is an LLM agent and how it differs from plain chat."
"Please think deeply to show your reasoning steps and summary.")

result = agent.run_sync(query)
print(result.output)

## 2) Add tools to the agent

We now give the model several tiny tools it can call. We will create the tools first:

In [ ]:
def safe_calculator(expression: str) -> dict[str, Any]:
    try:
        tree = ast.parse(str(expression), mode="eval")
        allowed = (
            ast.Expression,
            ast.BinOp,
            ast.UnaryOp,
            ast.Add,
            ast.Sub,
            ast.Mult,
            ast.Div,
            ast.Mod,
            ast.Pow,
            ast.USub,
            ast.UAdd,
            ast.Constant,
        )
        for node in ast.walk(tree):
            if not isinstance(node, allowed):
                raise ValueError(f"Disallowed syntax: {type(node).__name__}")
        value = eval(compile(tree, filename="<calc>", mode="eval"), {"__builtins__": {}}, {})
        return {"expression": expression, "value": value}
    except Exception as exc:
        return {"expression": expression, "error": str(exc)}


def nm_to_ev(wavelength_nm: float) -> float:
    """Convert wavelength in nm to energy in eV."""
    energy_ev = wavelength_nm / 1240
    return energy_ev

def ev_to_nm(energy_ev: float) -> float:
    """Convert energy in eV to wavelength in nm."""
    wavelength_nm = energy_ev * 1240
    return wavelength_nm

def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert temperature from Celsius to Fahrenheit."""
    temp_fahrenheit = celsius * (9/5) + 32
    return temp_fahrenheit

def fahrenheit_to_celsius(fahrenheit: float) -> float:
    """Convert temperature from Fahrenheit to Celsius."""
    temp_celsius = (fahrenheit - 32) * (5/9)
    return temp_celsius

COURSE_DOCS = [
    {
        "title": "Section 1 - Knowledge graphs",
        "content": "What is a knowledge graph, and how to build one.",
    },
    {
        "title": "Section 2 - Self-driving labs",
        "content": "What is a self-driving lab, and how to use agents to control it.",
    },
    {
        "title": "Section 3 - LLM agents",
        "content": "What is an agent, tool use, and agent design patterns.",
    },
]


def search_course_docs(query: str, top_k: int = 3) -> dict[str, Any]:
    tokens = set(re.findall(r"[a-zA-Z0-9]+", str(query).lower()))
    scored = []
    for doc in COURSE_DOCS:
        text = f"{doc['title']} {doc['content']}".lower()
        doc_tokens = set(re.findall(r"[a-zA-Z0-9]+", text))
        scored.append((len(tokens & doc_tokens), doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    hits = [doc for score, doc in scored if score > 0][:top_k]
    return {"query": query, "hits": hits, "total_hits": len(hits)}


Now we register the tools we created.

In [ ]:
@dataclass
class LectureDeps:
    lecturer: str
    course: str

agent = Agent(
    model=OpenAIChatModel(LLM_MODEL, provider=openrouter_provider),
    instructions=(
        "You are a guest-lecture assistant for LLM agents. "
        "Use tools for facts/computation. Keep answers concise and explicit."
    )
)

@agent.tool
def get_lecture_context(ctx: RunContext[LectureDeps]) -> dict[str, str]:
    """Return metadata about this lecture setup."""
    return {"lecturer": ctx.deps.lecturer, "course": ctx.deps.course}

@agent.tool_plain
def calculator(expression: str) -> dict[str, Any]:
    """Safely evaluate arithmetic expressions."""
    return safe_calculator(expression)

@agent.tool_plain
def search_course_docs_tool(query: str, top_k: int = 3) -> dict[str, Any]:
    """Search mini in-memory course notes."""
    return search_course_docs(query=query, top_k=top_k)

@agent.tool_plain
def unit_conversion(value: float, from_unit: str, to_unit: str) -> float:
    """Convert units between nm<->eV and Celsius<->Fahrenheit."""
    from_unit = from_unit.lower()
    to_unit = to_unit.lower()

    if from_unit == "nm" and to_unit == "ev":
        return nm_to_ev(value)
    elif from_unit == "ev" and to_unit == "nm":
        return ev_to_nm(value)
    elif from_unit == "celsius" and to_unit == "fahrenheit":
        return celsius_to_fahrenheit(value)
    elif from_unit == "fahrenheit" and to_unit == "celsius":
        return fahrenheit_to_celsius(value)
    else:
        raise ValueError(f"Unsupported unit conversion: {from_unit} to {to_unit}")

deps = LectureDeps(lecturer="Tutorial Lecturer", course="Intro to LLM Agents")

print(f"Agent ready with {len(agent._function_toolset.tools)} tools\n")
print("Tools registered on agent:")
for tool_name in agent._function_toolset.tools:
    print(f"  {tool_name}")



**Let's see the agent in action.**

In [ ]:
query = (
    "Use tools to do the following tasks: "
    "(1) which section in our course covers self-driving labs?" \
    "(2) What is the energy in eV for a wavelength of 500nm?" \
    "(3) What is 349 * 334?"
    "(4) Who is the lecturer of this course?"
)

result = agent.run_sync(query, deps=deps)
print("---"*3)
print(result.output)


## 3) Read the trace, not just the final answer

`user prompt -> tool call -> tool return -> final response`.

You can go through this manually or by looking at the logfire.

In [ ]:
def normalize_args(args_obj: Any) -> Any:
    if hasattr(args_obj, "as_dict"):
        try:
            return args_obj.as_dict()
        except Exception:
            pass
    return args_obj


trace_rows = []
for msg in result.all_messages():
    for part in msg.parts:
        kind = getattr(part, "part_kind", type(part).__name__)
        row = {"part_kind": kind}

        if kind == "user-prompt":
            row["content"] = getattr(part, "content", "")
        elif kind == "text":
            row["content"] = getattr(part, "content", "")
        elif kind == "tool-call":
            row["tool_name"] = getattr(part, "tool_name", "")
            row["args"] = normalize_args(getattr(part, "args", {}))
        elif kind == "tool-return":
            row["tool_name"] = getattr(part, "tool_name", "")
            row["content"] = getattr(part, "content", "")

        trace_rows.append(row)

for i, row in enumerate(trace_rows, start=1):
    print(f"[{i}] {row['part_kind']}")
    for k, v in row.items():
        if k != "part_kind":
            if isinstance(v, (dict, list)):
                print(f"  - {k}: {json.dumps(v, ensure_ascii=False)}")
            else:
                print(f"  - {k}: {v}")


## 4) Multi-agent mini demo

We split roles:
- Planner agent → decides which execution agents to use
- Execution agents → uses their own tool repertoire

**First, we create the expert agents with their custom tools**:

In [ ]:
teacher_agent = Agent(
    model=OpenAIChatModel(
        LLM_MODEL,
        provider=openrouter_provider
    ),
    instructions="You are the teacher who performs calculations."
)

course_coordinator_agent = Agent(
    model=OpenAIChatModel(
        LLM_MODEL,
        provider=openrouter_provider),
    instructions=(
        "You are the course coordinator who knows the course content."
    )
)

teacher_agent.tool_plain(calculator)
teacher_agent.tool_plain(unit_conversion)
course_coordinator_agent.tool_plain(search_course_docs_tool)
course_coordinator_agent.tool(get_lecture_context)

**Next we create the planner agent and expose the expert agents as tools.**

In [ ]:
planning_agent = Agent(
    model=OpenAIChatModel(LLM_MODEL, provider=openrouter_provider),
    instructions=(
        "You are the planning agent. Break tasks into steps and choose the most suitable agent to execute the task."
    )
)

@planning_agent.tool_plain
async def teacher(query: str) -> str:
    """Delegate calculation-related questions to the teacher agent."""
    results = await teacher_agent.run(query)
    return results.output

@planning_agent.tool_plain
async def docs_specialist(query: str) -> str:
    """Delegate course questions to the course coordinator agent."""
    results = await course_coordinator_agent.run(query)
    return results.output

In [ ]:
query = ("What is the energy (eV) of a 300nm wavelength laser")
result = planning_agent.run_sync(
    query,
    deps=deps
)
print("---"*3)
print(result.output)

## 5) Multi-turn memory with `message_history`

We pass `turn1.all_messages()` into the next run and let the model continue from prior context.

This is also where caching happens.

In [ ]:
turn1 = agent.run_sync(
    "First, find which section discusses LLM agent systems in our course docs.",
    deps=deps,
)

turn2 = agent.run_sync(
    "Based on your previous answer, give me a 3-item pre-class checklist.",
    deps=deps,
    message_history=turn1.all_messages(),
)

print("---"*3)
print("Turn 1:")
print(turn1.output)
print("\nTurn 2 (with memory):")
print(turn2.output)


## 6) Guardrail LLM agent

`UsageLimits(request_limit=1)` restrict number of LLM API request.


In [ ]:
try:
    limited_result = agent.run_sync(
        "Use tools to get Toronto time and compute 99*13.",
        deps=deps,
        usage_limits=UsageLimits(request_limit=2),
    )
    print("---"*3)
    print(limited_result.output)
except UsageLimitExceeded as exc:
    print("---"*3)
    print("Guardrail triggered:")
    print(exc)

## Closing discussion

Questions:
- What kind of tool calls would you allow without human approval?
- Where should tool use be restricted?
- What would a useful failure report look like for human to investigate?
- What would you log in your agentic system (e.g. within tools)?

Some more things you can explore:
1. Replace in-memory docs with retrieval over external notes.
2. Add a human approval layer for high-risk tool calls.
3. Track token/cost trends over a batch of prompts.
